# Stage 1B — Bangla single-task seed-42 pilot
This guarded Kaggle launcher runs exactly one selected three-class Bangla model. It uses the complete audited training split, selects by validation macro F1, writes validation-only artifacts, and never reads the test split. Results are marked **PILOT — NOT FINAL TEST RESULT**.

In [ ]:
from pathlib import Path
import json, subprocess, sys

RUN_HEAVY = False
MODEL_TO_RUN = 'mdistilbert'  # Allowed: 'mdistilbert' or 'xlmr'.
ALLOWED_MODELS = {
    'mdistilbert': 'pilot_bn_mdistilbert_single_seed42.json',
    'xlmr': 'pilot_bn_xlmr_single_seed42.json',
}

if MODEL_TO_RUN not in ALLOWED_MODELS:
    raise ValueError(f'MODEL_TO_RUN must be one of {tuple(ALLOWED_MODELS)}')

def find_repo():
    candidates = [Path('/kaggle/working/Capstone-Project')]
    candidates.extend(Path('/kaggle/input').glob('*/Capstone-Project'))
    candidates.extend(path for path in Path('/kaggle/input').glob('*') if (path / 'corrected_pipeline').is_dir())
    required_configs = {
        'pilot_bn_mdistilbert_single_seed42.json',
        'pilot_bn_xlmr_single_seed42.json',
    }
    for candidate in candidates:
        config_source = candidate / 'corrected_pipeline' / 'config.py'
        has_pilot_configs = all((candidate / 'configs' / name).is_file() for name in required_configs)
        if (candidate / 'corrected_pipeline' / 'runner.py').is_file() and config_source.is_file() and has_pilot_configs:
            source = config_source.read_text(encoding='utf-8')
            if '{"smoke", "pilot", "full"}' in source:
                return candidate
    raise FileNotFoundError('Attach the updated Stage 1B-capable Capstone-Project repository to this Kaggle notebook.')

if not RUN_HEAVY:
    print('PILOT — NOT FINAL TEST RESULT')
    print(f'Guard active for {MODEL_TO_RUN}: no model, checkpoint, or dataset was loaded.')
else:
    repo = find_repo()
    # Lightweight syntax and unit checks must pass before any training command.
    subprocess.run([sys.executable, '-m', 'compileall', '-q', 'corrected_pipeline', 'tests_stage1a'], cwd=repo, check=True)
    subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests_stage1a', '-v'], cwd=repo, check=True)

    config_path = repo / 'configs' / ALLOWED_MODELS[MODEL_TO_RUN]
    config = json.loads(config_path.read_text(encoding='utf-8'))
    assert config['run_kind'] == 'pilot'
    assert config['result_status'] == 'PILOT — NOT FINAL TEST RESULT'
    assert config['dataset']['languages'] == ['bangla']
    assert set(config['dataset']['paths']) == {'bn_train', 'bn_validation'}
    assert set(config['dataset']['hashes']) == {'bn_train', 'bn_validation'}
    assert config['dataset']['path_base'] == 'repository_root'
    assert config['auxiliary_labels'] == {'mode': 'none', 'proxy_auxiliary_labels': False}
    assert config['training']['random_seed'] == 42
    assert config['training']['selection_metric'] == 'validation_macro_f1'
    assert config['execution']['evaluate_test'] is False
    assert config['execution']['allow_overwrite'] is False
    dataset_paths = {key: repo / value for key, value in config['dataset']['paths'].items()}
    missing = [str(path) for path in dataset_paths.values() if not path.is_file()]
    if missing:
        raise FileNotFoundError('Missing audited repository dataset files: ' + ', '.join(missing))
    if Path(config['execution']['output_directory']).exists():
        raise FileExistsError('Output already exists; refusing to overwrite: ' + config['execution']['output_directory'])

    import torch
    if not torch.cuda.is_available():
        raise RuntimeError('Enable a Kaggle GPU before running the Stage 1B pilot.')
    print(f'PILOT — NOT FINAL TEST RESULT: running only {MODEL_TO_RUN}')
    subprocess.run([sys.executable, '-m', 'corrected_pipeline.runner', '--config', str(config_path)], cwd=repo, check=True)